In [1]:
path_pattern = "/workspaces/mads-siads-699-winter-2026-capstone/notebooks/data/source_data/initial_dataset/*.parquet"

In [2]:
import polars as pl
import glob

all_files = sorted(glob.glob(path_pattern))
first_five = all_files[:10]
if first_five:
    df_deps = pl.read_parquet(first_five)
    print(f"Loaded {len(first_five)} files.")

Loaded 10 files.


In [3]:
df_deps.shape

(26055035, 8)

In [4]:
df_deps.head(30)

SnapshotAt,package_name,package_version,package_published_at,dep_name,dep_version,MinimumDepth,github_repo
datetime[μs],str,str,datetime[μs],str,str,i64,str
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""certifi""","""2022.12.7""",3,"""jdelic/12factor-vault"""
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""charset-normalizer""","""3.1.0""",3,"""jdelic/12factor-vault"""
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""django-dbconn-retry""","""0.1.7""",1,"""jdelic/12factor-vault"""
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""hvac""","""1.1.0""",1,"""jdelic/12factor-vault"""
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""idna""","""3.4.0""",3,"""jdelic/12factor-vault"""
…,…,…,…,…,…,…,…
2023-04-10 21:01:49.219309,"""2captcha-python""","""1.2.0""",null,"""urllib3""","""1.26.15""",2,"""2captcha/2captcha-python"""
2023-04-10 21:01:49.219309,"""2ppy""","""0.4.0""",null,"""jpype1""","""1.3.0""",1,"""tuprolog/2ppy"""
2023-04-10 21:01:49.219309,"""a-cv-imwrite-imread-plus""","""0.11.0""",null,"""certifi""","""2022.12.7""",2,"""hansalemaos/a_cv_imwrite_imrea…"


In [5]:
path_pattern = "/workspaces/mads-siads-699-winter-2026-capstone/notebooks/data/source_data/pypi_scorecards/*.parquet"
all_files = sorted(glob.glob(path_pattern))
first_five = all_files[:10]
if first_five:
    df_scorecard = pl.read_parquet(first_five)
    print(f"Loaded {len(first_five)} files.")

Loaded 10 files.


In [6]:
df_scorecard.head()

scorecard_date,repo_name,repo_commit,scorecard_version,aggregate_score,check_name,check_score
date,str,str,str,f64,str,i64
2023-10-30,"""github.com/keybase/pykeybasebo…","""38c84aac41fc03d4d2693fb316288d…","""v4.13.1-26-g478f347e""",2.5,"""Code-Review""",6
2023-10-30,"""github.com/keybase/pykeybasebo…","""38c84aac41fc03d4d2693fb316288d…","""v4.13.1-26-g478f347e""",2.5,"""Pinned-Dependencies""",-1
2023-10-30,"""github.com/paweltroka/signalr-…","""effa7347016174e460d3c3e3f2a9d0…","""v4.13.1-26-g478f347e""",2.5,"""Binary-Artifacts""",10
2023-10-30,"""github.com/kaixuyang/lunarcale…","""888497987ffe7fecca94f544c03e56…","""v4.13.1-26-g478f347e""",3.0,"""Dangerous-Workflow""",-1
2023-10-30,"""github.com/jjmontesl/codenamiz…","""5af5f8bca3b39d18806a051fd112ea…","""v4.13.1-26-g478f347e""",3.0,"""CII-Best-Practices""",0


In [7]:
df_scorecard['repo_name'].unique()

repo_name
str
"""github.com/gri38/hyperlink_pre…"
"""github.com/iotic-labs/iotics-i…"
"""github.com/gamcil/clinker"""
"""github.com/jazzband/django-use…"
"""github.com/snehankekre/streaml…"
…
"""github.com/thecapypara/riptide…"
"""github.com/vladimirs-git/forti…"
"""github.com/inveniosoftware/bab…"


In [8]:
df_deps = df_deps.with_columns(
    pl.col("github_repo").str.replace("https://", "").str.replace("github.com/", ""),
    pl.col("package_published_at").cast(pl.Datetime)
)

dtype = df_scorecard.schema.get("scorecard_date")
if dtype is not None and "Date" in str(dtype):
    df_scorecard = df_scorecard.with_columns(
        pl.col("repo_name").str.replace("https://", "").str.replace("github.com/", ""),
        pl.col("scorecard_date").cast(pl.Datetime)
    )
else:
    df_scorecard = df_scorecard.with_columns(
        pl.col("repo_name").str.replace("https://", "").str.replace("github.com/", ""),
        pl.col("scorecard_date").str.to_datetime().cast(pl.Datetime)
    )


df_deps = df_deps.sort("package_published_at")
df_scorecard = df_scorecard.sort("scorecard_date")


df_joined = df_deps.join_asof(
    df_scorecard,
    left_on="package_published_at",
    right_on="scorecard_date",
    by_left="github_repo",
    by_right="repo_name",
    strategy="backward"
)

/tmp/ipykernel_19642/3577651225.py:23: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df_joined = df_deps.join_asof(


In [16]:
df_joined

SnapshotAt,package_name,package_version,package_published_at,dep_name,dep_version,MinimumDepth,github_repo,scorecard_date,repo_commit,scorecard_version,aggregate_score,check_name,check_score
datetime[μs],str,str,datetime[μs],str,str,i64,str,datetime[μs],str,str,f64,str,i64
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""certifi""","""2022.12.7""",3,"""jdelic/12factor-vault""",null,null,null,null,null,null
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""charset-normalizer""","""3.1.0""",3,"""jdelic/12factor-vault""",null,null,null,null,null,null
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""django-dbconn-retry""","""0.1.7""",1,"""jdelic/12factor-vault""",null,null,null,null,null,null
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""hvac""","""1.1.0""",1,"""jdelic/12factor-vault""",null,null,null,null,null,null
2023-04-10 21:01:49.219309,"""12factor-vault""","""0.1.23""",null,"""idna""","""3.4.0""",3,"""jdelic/12factor-vault""",null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…
2023-10-23 21:01:31.784324,"""ccs-digitalmarketplace-utils""","""62.3.2""",2023-10-23 11:18:56,"""urllib3""","""1.26.18""",2,"""brickendon-dmp1-5/digitalmarke…",null,null,null,null,null,null
2023-10-23 21:01:31.784324,"""ccs-digitalmarketplace-utils""","""62.3.2""",2023-10-23 11:18:56,"""werkzeug""","""3.0.0""",2,"""brickendon-dmp1-5/digitalmarke…",null,null,null,null,null,null
2023-10-23 21:01:31.784324,"""ccs-digitalmarketplace-utils""","""62.3.2""",2023-10-23 11:18:56,"""workdays""","""1.4.0""",1,"""brickendon-dmp1-5/digitalmarke…",null,null,null,null,null,null


In [ ]:
# below is another method of joining but it's an aggregation and it didn't properly preserve the data 

In [9]:
# df_scorecard = df_scorecard.with_columns(
#     pl.col("repo_name").str.replace(r"^github\.com/", "").alias("match_repo")
# )
# df_deps = df_deps.with_columns(
#     pl.col("github_repo").alias("match_repo")
# )

In [10]:
# df_scorecard_collapsed = (
#     df_scorecard
#     .group_by("match_repo")
#     .agg([
#         pl.col("aggregate_score").first(),  # Keep the main score
#         # Optional: Get specific scores as columns
#         pl.col("check_score").filter(pl.col("check_name") == "Code-Review").first().alias("score_code_review"),
#         pl.col("check_score").filter(pl.col("check_name") == "Maintained").first().alias("score_maintained")
#     ])
# )


In [11]:
# df_scorecard_collapsed.head()

In [12]:
# joined_df = df_deps.join(df_scorecard_collapsed, on="match_repo", how="left")

In [13]:
# joined_df